In [10]:
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from langchain.chat_models import init_chat_model
from typing import TypedDict, Literal
from pydantic import BaseModel, Field
from IPython.display import Image, display

llm = init_chat_model(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

class JokeState(TypedDict):
    joke: str
    topic: str
    feedback: str
    funny_or_not: str

class Feedback(BaseModel):
    grade: Literal["funny", "not funny"] = Field(description="判断笑话是否好笑")
    feedback: str = Field(description="如果不好笑，给出改进建议")

evaluator = llm.with_structured_output(Feedback)

def generate_node(state: JokeState):
    if state.get("feedback"):
        msg = llm.invoke(
            f"请根据反馈「{state['feedback']}」，给我讲一个关于{state['topic']}的笑话"
        )
    else:
        msg = llm.invoke(f"给我讲一个关于{state['topic']}的笑话")
    return {"joke": msg.content}

def evaluate_node(state: JokeState):
    grade = evaluator.invoke(f"请评价这个笑话是否好笑：{state['joke']}")
    return {"funny_or_not": grade.grade, "feedback": grade.feedback}

def route_joke(state: JokeState):
    if state["funny_or_not"] == "funny":
        return "Accept"
    return "Reject"

builder = StateGraph(JokeState)
builder.add_node("generate", generate_node)
builder.add_node("evaluate", evaluate_node)
builder.add_edge(START, "generate")
builder.add_edge("generate", "evaluate")
builder.add_conditional_edges("evaluate", route_joke, {
    "Accept": END,
    "Reject": "generate",
})

graph = builder.compile()
# display(Image(graph.get_graph().draw_mermaid_png()))


In [13]:
state = graph.invoke({"topic": "小猫"})
print(state)
# print("评级:", state["funny_or_not"])
# print("反馈:", state["feedback"])
# print("笑话:", state["joke"])
# for msg in state["messages"]:
#     msg.pretty_print()

{'joke': '一只猫走进酒吧，往吧台上一坐，掏出平板电脑开始打字。\n\n酒保凑过去看了一眼屏幕，问：“你在搜什么？”\n\n猫头也不抬：“‘如何优雅地向酒保讨要第二杯牛奶然后赖账’。”\n\n酒保愣了一下：“……你第一杯还没点呢。”\n\n猫把平板一扣，叹了口气：“我知道。我在做市场调研。”', 'topic': '小猫', 'feedback': '结构完整，三段式反转到位：先建立"猫点牛奶"的荒诞前提，中段用"第一杯还没点"制造小反转，结尾"做市场调研"把"白嫖"行为升格成商业行为，荒诞感叠满，且猫的傲娇人设和酒保的懵圈反应对比鲜明。唯一可优化处：中间台词可以再压缩半句，让节奏更紧。', 'funny_or_not': 'funny'}
